## DNN Serving System

In [ ]:
import torch
import os

### Scheduler

In [ ]:
import torch
from torchvision.models import resnet18
from Simulator.simulator import TOGSimulator
from PyTorchSimFrontend import extension_config

device = torch.device("npu:0")
config = "/workspace/PyTorchSim/tutorial/session1/togsim_configs/togsim_config_timing_only.yml"

model = resnet18().eval()
input = torch.randn(1, 3, 224, 224).to(device=device)
opt_fn = torch.compile(dynamic=False)(model.to(device, memory_format=torch.channels_last))

# Run scheduler
with TOGSimulator(config_path=config):
    torch.npu.launch_model(opt_fn, input, stream_index=0, timestamp=0)

print("ResNet18 Simulation Done")

### Load Generator

In [ ]:
from Scheduler.scheduler import poisson_request_generator

model0_lambda = 5.0
max_time_msec = 1000.0

target_model1 = resnet18().eval()

device = torch.device("npu:0")
config = "/workspace/PyTorchSim/tutorial/session1/togsim_configs/togsim_config_timing_only.yml"
opt_model0 = torch.compile(target_model1.to(device=device, memory_format=torch.channels_last), dynamic=False)

events = []
x = torch.randn(1, 3, 224, 224, device=device)
for t in poisson_request_generator(model0_lambda, max_msec_time=max_time_msec):
    events.append((t, 0, opt_model0, (x,)))  # stream_index 0 → queue / partition 0

with TOGSimulator(config_path=config):
    for t_msec, stream_index, model, args in events:
        torch.npu.launch_model(
            model,
            *args,
            stream_index=stream_index,
            timestamp=int(t_msec),
        )